In [4]:
import numpy as np
import pandas as pd
import random
from xgboost import XGBRegressor

In [ ]:
class CausalContextualBandit:
    def __init__(self, treatments, epsilon=0.15):
        """
        treatments: Lista de assuntos possíveis (ex: ['pix', 'pagamento', 'seguro', 'investimento'])
        epsilon: Taxa de exploração (ex: 15% das vezes escolhe aleatoriamente para aprender)
        """
        self.treatments = treatments
        self.control_name = 'controle' # O braço silencioso
        self.epsilon = epsilon
        
        # Um modelo para cada tratamento. Ele vai prever o UPLIFT (CATE) contra o controle.
        self.models = {
            trt: XGBRegressor(n_estimators=50, max_depth=3, learning_rate=0.1) 
            for trt in treatments
        }
        
        # Flag para saber se os modelos já foram treinados pela primeira vez
        self.is_trained = {trt: False for trt in treatments}
    
    @property
    def arms(self):
        return self.treatments + [self.control_name]
    
    def propensities(self, context_features):
        """
        Retorna a distribuição de probabilidade COMPLETA sobre os braços para esse contexto.
        
        É essa distribuição inteira que precisa ser logada (não só a probabilidade do
        braço sorteado): para estimar o uplift de um tratamento contra o controle, o
        treino precisa saber qual era a chance daquele tratamento até para os clientes
        que caíram no controle.
        """
        arms = self.arms
        
        # Base: todo braço tem chance epsilon/n_arms de ser sorteado na exploração
        probs = {arm: self.epsilon / len(arms) for arm in arms}
        
        # Qual braço a explotação escolheria?
        uplifts = {}
        for trt in self.treatments:
            if self.is_trained[trt]:
                # Prever o incremento (Uplift) que essa mensagem gera
                x = np.asarray(context_features, dtype=float).reshape(1, -1)
                uplifts[trt] = self.models[trt].predict(x)[0]
            else:
                # Se ainda não tem dados, assume zero para forçar exploração
                uplifts[trt] = 0.0
        
        best_treatment = max(uplifts, key=uplifts.get)
        
        # A DECISÃO CAUSAL: Se nenhuma mensagem gerar aumento de chance (uplift <= 0),
        # a melhor ação é NÃO MANDAR NADA (Controle).
        greedy_arm = best_treatment if uplifts[best_treatment] > 0 else self.control_name
        
        # O braço guloso acumula a massa restante (1 - epsilon)
        probs[greedy_arm] += 1 - self.epsilon
        
        return probs
        
    def recommend(self, context_features):
        """
        Recebe o contexto (features do cliente) e retorna a Ação (Assunto) e o dicionário
        de propensões de TODOS os braços (para logar).
        """
        probs = self.propensities(context_features)
        arms = list(probs.keys())
        
        # Sorteia direto da distribuição: garante que a propensão logada é exatamente
        # a probabilidade com que a ação foi escolhida.
        chosen_arm = np.random.choice(arms, p=[probs[a] for a in arms])
        
        return chosen_arm, probs

    def train_batch(self, batch_data):
        """
        batch_data: DataFrame contendo o log do que aconteceu após os 7 dias.
        Colunas esperadas: features do contexto, 'action', 'converted' e uma coluna de
        propensão por braço, nomeada 'p_<braço>' (incluindo 'p_controle').
        """
        prop_cols = [f'p_{arm}' for arm in self.arms]
        log_cols = ['action', 'converted'] + prop_cols
        
        # Para cada assunto, treinamos o modelo usando seus dados + os dados do grupo de controle
        for trt in self.treatments:
            # Filtra apenas quem recebeu ESTE tratamento ou o CONTROLE
            df_subset = batch_data[batch_data['action'].isin([trt, self.control_name])].copy()
            
            if len(df_subset) < 10: # Só treina se tiver o mínimo de dados
                continue
            
            # Variável W (1 se foi Tratamento, 0 se foi Controle)
            w = np.where(df_subset['action'] == trt, 1, 0)
            y = df_subset['converted'].values
            
            # -------------------------------------------------------------
            # A PROPENSÃO CORRETA: p = P(W=1 | X)
            # Não é a propensão da ação tomada. Como aqui só existem duas opções
            # (este tratamento ou o controle), renormalizamos dentro do par:
            #     p = P(trt) / (P(trt) + P(controle))
            # Essa conta vale para TODA linha do subset, tratada ou controle, que é
            # exatamente o que a fórmula do Y* exige.
            # -------------------------------------------------------------
            p_trt = df_subset[f'p_{trt}'].values
            p_ctrl = df_subset[f'p_{self.control_name}'].values
            p = p_trt / (p_trt + p_ctrl)
            
            # Proteção matemática para não dividir por zero
            p = np.clip(p, 0.01, 0.99)
            
            # -------------------------------------------------------------
            # O CORAÇÃO DO ALGORITMO: O RESULTADO TRANSFORMADO (Y*)
            # Isso transforma a conversão simples em "Uplift Incremental"
            # Fórmula do Y*: Y * ((W / p) - ((1 - W) / (1 - p)))
            # -------------------------------------------------------------
            y_star = y * ( (w / p) - ((1 - w) / (1 - p)) )
            
            # Features (Remove colunas de log)
            X = df_subset.drop(columns=log_cols).values
            
            # Treina o modelo XGBoost para prever o Y*
            self.models[trt].fit(X, y_star)
            self.is_trained[trt] = True
            
        print("Modelos de Uplift atualizados com sucesso!")

# ==========================================
# SIMULANDO O USO EM PRODUÇÃO
# ==========================================

# 1. Instanciamos nosso Agente Causal
treatments = ['pix', 'pagamento', 'seguro', 'investimento']
bandit = CausalContextualBandit(treatments=treatments, epsilon=0.20)

# 2. Quando o cliente abre a conta (Dia 1)
cliente_features = [25, 3500.0, 1] # Ex: Idade, Renda, Usa iOS (1=Sim)
assunto, propensoes = bandit.recommend(cliente_features)

print(f"O motor escolheu: {assunto} (Probabilidade de escolha: {propensoes[assunto]:.2f})")
# AQUI VOCÊ PASSA 'assunto' PARA O SEU LLM GERAR A MENSAGEM (ou não gera nada se for controle).

# 3. Logamos isso no Banco de Dados (Ex: enviamos pro Kafka/Data Warehouse)
# Note que logamos a propensão de TODOS os braços, não só a do escolhido.
# log = {'idade': 25, 'renda': 3500, 'ios': 1, 'action': assunto,
#        'p_pix': ..., 'p_pagamento': ..., 'p_seguro': ..., 'p_investimento': ...,
#        'p_controle': ..., 'converted': ?}

# ... 7 DIAS SE PASSAM ...

# 4. Job Diário pega os clientes que bateram 7 dias e atualiza o modelo em Batch
# Simulação de um log do banco de dados (retorno dos 7 dias):
dados_historicos = pd.DataFrame({
    'idade': np.random.randint(18, 65, 1000),
    'renda': np.random.uniform(1000, 15000, 1000),
    'ios': np.random.randint(0, 2, 1000),
    'action': np.random.choice(['controle', 'pix', 'pagamento', 'seguro', 'investimento'], 1000),
    # Supondo que foi totalmente exploratório no início: 1/5 para cada braço.
    # Renormalizado no treino, isso vira p = 0.2 / (0.2 + 0.2) = 0.5.
    'p_pix': [0.2] * 1000,
    'p_pagamento': [0.2] * 1000,
    'p_seguro': [0.2] * 1000,
    'p_investimento': [0.2] * 1000,
    'p_controle': [0.2] * 1000,
    'converted': np.random.randint(0, 2, 1000) # 1 se ativou, 0 se não ativou
})

print("\nRodando o job de atualização (Batch)...")
bandit.train_batch(dados_historicos)

# 5. Nova recomendação (Dia 8) com o modelo mais inteligente
novo_cliente = [45, 12000.0, 0] # Cliente mais velho, renda alta, Android
assunto, propensoes = bandit.recommend(novo_cliente)
print(f"Para o novo cliente, o motor escolheu: {assunto}")